In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType
from pyspark.sql.functions import trim

In [0]:
SparkSession.builder.appName("Join_gold").getOrCreate()


simples_data_schema = StructType([
    StructField("DocumentNumber", StringType(), True),
    StructField("OPÇÃO PELO SIMPLES", StringType(), True),
    StructField("DATA DE OPÇÃO PELO SIMPLES", StringType(), True),
    StructField("DATA DE EXCLUSÃO DO SIMPLES", StringType(), True),
    StructField("MEI", StringType(), True),
    StructField("DATA DE OPÇÃO PELO MEI", StringType(), True),
    StructField("DATA DE EXCLUSÃO DO MEI", StringType(), True),    
])

In [0]:
df_mei = (spark.read
    .schema(simples_data_schema)
    .format("csv") 
    .option("sep", ";")
    .load("/Volumes/workspace/cnpjs_schema/simples_data")
).select("DocumentNumber", "MEI")

df_mei = df_mei.replace({"S":"1","N":"0"}, subset=["MEI"])

In [0]:
establishments_silver_path = "workspace.establishments_silver.silver_establishments"
enterprise_silver_path = "workspace.enterprises_silver.silver_enterprise"
tax_regime_silver = "workspace.tax_regime_silver.tax_regime_data"

In [0]:
df_establishments = spark.read.table(establishments_silver_path)
df_enterprises = spark.read.table(enterprise_silver_path)
df_tax_regime = spark.read.table(tax_regime_silver)

In [0]:
df_gold = df_establishments.join(df_enterprises, on='DocumentNumber', how='left') \
.join(df_mei, on="DocumentNumber", how='left') \
.join(df_tax_regime, on='CNPJ', how='left') 

In [0]:
df_gold.write.mode("overwrite").format("delta").option("OverwriteSchema", "true").saveAsTable("workspace.cnpj_gold.gold_data")

In [0]:
spark.read.table("workspace.cnpj_gold.gold_data").display()